In [2]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CLIP × ViLT — Cross-Attention Fusion — Fakeddit 2-Way (Real vs Fake)      ║
# ║  Version : v9  (built directly on clip-vilt-fakeddit-2way-fixed.ipynb)      ║
# ║                                                                              ║
# ║  TASK    : 2-WAY ONLY  (3-way & 6-way labels completely excluded)           ║
# ║  DATASET : Fakeddit balanced samples (metadata disabled)                    ║
# ║  SPLIT   : 80% train / 10% val / 10% test, grouped by normalized title      ║
# ║  CV      : 5-Fold                                                            ║
# ║  EPOCHS  : 10  (EarlyStopping on leakage-penalized val score)               ║
# ║  TARGET  : ≥ 90% Acc,  F1 ≈ Prec ≈ Rec ≈ Acc,  OvfGap < 5%               ║
# ║  PLATFORM: Kaggle GPU  (T4 / P100)                                           ║
# ╠══════════════════════════════════════════════════════════════════════════════╣
# ║  WHAT CHANGED vs the original 2-way fixed notebook                          ║
# ║  ─────────────────────────────────────────────────────────────────────────  ║
# ║  [1]  NUM_SAMPLES increased when available to reduce memorization           ║
# ║  [2]  METADATA disabled completely: no subreddit/comments/source shortcuts  ║
# ║  [3]  Duplicate normalized titles are grouped across train/val/test         ║
# ║  [4]  Selective unfreezing reduced to last 1 CLIP + ViLT block              ║
# ║  [5]  Compact fusion/classifier head with stronger dropout/weight decay     ║
# ║  [6]  EarlyStopping penalizes large train/validation accuracy gaps          ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ── Cell 1: Install Dependencies ───────────────────────────────────────────────
import sys, subprocess

pkgs = [
    "transformers>=4.40.0",
    "pillow",
    "tabulate",
    "scikit-learn",
    "timm",
]
for i, p in enumerate(pkgs, 1):
    print(f"[{i}/{len(pkgs)}] {p} ...", end=" ", flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])
    print("✓")

import transformers, torch
print(f"\ntransformers {transformers.__version__} | torch {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ── Cell 2: Imports & Configuration ───────────────────────────────────────────
import os, gc, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from PIL import Image
from transformers import (
    CLIPProcessor, CLIPModel,
    ViltProcessor, ViltModel,
    get_cosine_schedule_with_warmup,
)
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
)
from sklearn.utils.class_weight import compute_class_weight
from tabulate import tabulate

warnings.filterwarnings("ignore")

# ═══════════════════════════════ CONFIGURATION ════════════════════════════════
SEED          = 42
NUM_SAMPLES   = 40_000    # More real samples reduce memorization and stabilize 2-way CV.
NUM_FOLDS     = 5         # Fewer, larger folds give a more reliable non-leaky estimate.
EPOCHS        = 10        # Lower max epochs; early stopping handles hard folds.
BATCH_SIZE    = 16        # per device; accum ×2 → eff. batch = 32
ACCUM_STEPS   = 2         # [9] gradient accumulation
NUM_CLASSES   = 2         # 2-WAY ONLY

# Learning rates (layer-wise)
LR_HEAD       = 1e-4      # projection + classifier
LR_UNFREEZE   = 2e-6      # last transformer block of both backbones
WEIGHT_DECAY  = 0.10
WARMUP_RATIO  = 0.15      # [8] 15% of steps for cosine warm-up
DROPOUT_RATE  = 0.50
PATIENCE      = 3         # [10] early stop on leakage-penalized val score
LABEL_SMOOTH  = 0.08
OVERFIT_PENALTY = 0.35

# Architecture
CLIP_MODEL      = "openai/clip-vit-base-patch32"
VILT_MODEL      = "dandelin/vilt-b32-mlm"
CLIP_VISION_DIM = 768    # ViT-B/32 patch token dim
VILT_TEXT_DIM   = 768    # ViLT-B/32 text token dim
FUSION_DIM      = 256
FUSION_HEADS    = 4
FUSION_LAYERS   = 1       # Keep the fusion head compact to reduce fold memorization.
UNFREEZE_BLOCKS = 1       # [6] last N blocks unfrozen

OUTPUT_DIR = "/kaggle/working"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = (DEVICE == "cuda")

print(f"Device        : {DEVICE}")
print(f"Task          : 2-WAY ONLY  (3-way & 6-way EXCLUDED)")
print(f"Samples       : {NUM_SAMPLES:,}  (balanced)")
print(f"CV            : {NUM_FOLDS}-fold grouped by normalized title  |  Split: 80 / 10 / 10")
print(f"Epochs        : {EPOCHS}  (EarlyStopping patience={PATIENCE})")
print(f"Metadata      : DISABLED  (headline text + image pixels only)")
print(f"Backbone      : CLIP ViT-B/32 + ViLT-B/32")
print(f"Unfreeze      : last {UNFREEZE_BLOCKS} blocks per backbone  LR={LR_UNFREEZE}")
print(f"Fusion        : {FUSION_DIM}-d  {FUSION_HEADS}h  {FUSION_LAYERS}L + StochasticDepth")
print(f"Loss          : CrossEntropy + class weights + label smoothing")
print("=" * 70)

# ── Cell 3: Fakeddit Loader — 2-way ONLY, no metadata leakage ─────────────────
#
#  The previous v9 metadata branch could learn dataset shortcuts such as
#  subreddit/source patterns and engagement artifacts. This version trains only
#  from the article headline text and image pixels. It also groups duplicated
#  normalized titles so the same headline cannot appear in both train and test.

def _is_fakeddit_2way(path):
    try:
        sep = "\t" if path.suffix == ".tsv" else ","
        hdr = pd.read_csv(path, sep=sep, nrows=0)
        hdr.columns = hdr.columns.str.strip()
        return "2_way_label" in hdr.columns
    except Exception:
        return False


def _normalize_split_text(text):
    return " ".join(str(text).lower().strip().split())


def load_fakeddit(n_samples=NUM_SAMPLES):
    """Load Fakeddit: 2-way labels ONLY, no metadata features."""
    print("🔍 Scanning /kaggle/input for Fakeddit dataset (2-way)...\n")
    for ext in ("*.tsv", "*.csv"):
        for p in sorted(Path("/kaggle/input").rglob(ext)):
            if _is_fakeddit_2way(p):
                print(f"  ✓ Found: {p}")
                return _load_file(p, n_samples)
            print(f"  ✗ Skip (no 2_way_label): {p.name}")
    raise FileNotFoundError(
        "❌ No Fakeddit file with 2_way_label found in /kaggle/input.\n"
        "   Add the Fakeddit dataset via '+ Add Data' on Kaggle.\n"
        "   Required columns: 2_way_label, clean_title/title, id."
    )


def _load_file(path, n_samples):
    sep = "\t" if path.suffix == ".tsv" else ","
    df  = pd.read_csv(path, sep=sep, low_memory=False)
    df.columns = df.columns.str.strip()
    print(f"  Columns: {list(df.columns[:15])} ...")

    # ── Text ──────────────────────────────────────────────────────────────────
    text_col = next(
        (c for c in ["clean_title", "title", "text"] if c in df.columns), None)
    if text_col is None:
        raise ValueError(f"No text column found. Available: {list(df.columns[:10])}")
    id_col = "id" if "id" in df.columns else df.columns[0]

    # ── 2-way label ONLY ──────────────────────────────────────────────────────
    df["label"] = df["2_way_label"].astype(int)
    df = df[df["label"].isin([0, 1])].copy()   # strict: drop anything not 0/1

    # ── Clean text ─────────────────────────────────────────────────────────────
    df["text"] = (df[text_col].astype(str).str.strip()
                              .str[:200]
                              .replace("", "[No text]"))
    df["split_group"] = df["text"].map(_normalize_split_text)

    # ── Images ────────────────────────────────────────────────────────────────
    img_dir = path.parent / "public_image_set"
    if not img_dir.exists():
        for d in path.parent.rglob("*"):
            if d.is_dir() and "image" in d.name.lower():
                img_dir = d; break
    if img_dir.exists():
        df["image_path"] = df[id_col].apply(lambda x: _find_img(x, img_dir))
        n_img = df["image_path"].notna().sum()
        print(f"  Images matched: {n_img}/{len(df)}")
    else:
        print("  ⚠️  No image directory — placeholder images used.")
        df["image_path"] = None

    # ── Keep needed columns ────────────────────────────────────────────────────
    keep = ["text", "label", "image_path", "split_group"]
    df = df[[c for c in keep if c in df.columns]].dropna(subset=["text", "label"])
    return _balance(df, n_samples)


def _find_img(img_id, img_dir):
    for ext in (".jpg", ".jpeg", ".png", ".gif", ".webp"):
        p = img_dir / f"{img_id}{ext}"
        if p.exists():
            return str(p)
    return None


def _balance(df, n):
    if len(df) == 0:
        raise ValueError("No valid samples after filtering!")
    half = n // 2
    real = df[df["label"] == 0]
    fake = df[df["label"] == 1]
    print(f"\n  Before balancing — Real: {len(real):,}  Fake: {len(fake):,}")
    n_real = min(half, len(real))
    n_fake = min(half, len(fake))
    k      = min(n_real, n_fake)   # equal class sizes
    out = (pd.concat([
               real.sample(k, random_state=SEED),
               fake.sample(k, random_state=SEED),
           ]).sample(frac=1, random_state=SEED)
            .reset_index(drop=True))
    print(f"  ✅ Balanced: {len(out):,} samples  (Real={k:,}  Fake={k:,})")
    print(f"  Split groups: {out['split_group'].nunique():,} unique normalized titles")
    if "image_path" in out.columns:
        print(f"  With real images: {out['image_path'].notna().sum():,}")
    return out


print("Fakeddit loader (2-way, no metadata) ready ✓")

# ── Cell 4: Dataset ────────────────────────────────────────────────────────────
_CLIP_PH = Image.new("RGB", (224, 224), (127, 127, 127))
_VILT_PH = Image.new("RGB", (384, 384), (127, 127, 127))

class FakedditDataset(Dataset):
    def __init__(self, df, clip_proc, vilt_proc):
        self.df        = df.reset_index(drop=True)
        self.clip_proc = clip_proc
        self.vilt_proc = vilt_proc

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        text  = str(row["text"])
        label = int(row["label"])

        # ── Image ─────────────────────────────────────────────────────────────
        img_path = row.get("image_path")
        try:
            raw = Image.open(img_path).convert("RGB") if img_path else None
        except Exception:
            raw = None
        clip_img = raw if raw is not None else _CLIP_PH
        vilt_img = raw if raw is not None else _VILT_PH

        # ── CLIP vision (patch tokens) ─────────────────────────────────────────
        clip_pv = self.clip_proc(
            images=clip_img, return_tensors="pt"
        )["pixel_values"].squeeze(0)                       # [3, 224, 224]

        # ── ViLT text + image ──────────────────────────────────────────────────
        vilt_out = self.vilt_proc(
            text=text, images=vilt_img,
            return_tensors="pt",
            padding="max_length", truncation=True, max_length=40,
        )

        item = {
            "clip_pixel_values" : clip_pv,
            "input_ids"         : vilt_out["input_ids"].squeeze(0),
            "token_type_ids"    : vilt_out["token_type_ids"].squeeze(0),
            "attention_mask"    : vilt_out["attention_mask"].squeeze(0),
            "vilt_pixel_values" : vilt_out["pixel_values"].squeeze(0),
            "labels"            : torch.tensor(label, dtype=torch.long),
        }
        if "pixel_mask" in vilt_out:
            item["pixel_mask"] = vilt_out["pixel_mask"].squeeze(0)
        return item


def collate_fn(batch):
    return {k: torch.stack([b[k] for b in batch]) for k in batch[0].keys()}


print("FakedditDataset (2-way text + image only) defined ✓")

# ── Cell 5: Cross-Attention Fusion with StochasticDepth ───────────────────────

class StochasticDepth(nn.Module):
    """Drop entire residual path with probability p (training only)."""
    def __init__(self, p=0.10):
        super().__init__()
        self.p = p

    def forward(self, x):
        if not self.training or self.p == 0.0:
            return x
        keep = torch.bernoulli(torch.tensor(1.0 - self.p)).item()
        return x * keep


class CrossAttentionLayer(nn.Module):
    """
    Bidirectional cross-attention: Vision ↔ Text.
    Vision attends to Text tokens, then Text attends back to Vision.
    StochasticDepth on both residual paths for regularisation.
    """
    def __init__(self, dim, num_heads, dropout, drop_path=0.10):
        super().__init__()
        self.v2t     = nn.MultiheadAttention(
            dim, num_heads, dropout=dropout, batch_first=True)
        self.t2v     = nn.MultiheadAttention(
            dim, num_heads, dropout=dropout, batch_first=True)
        self.ln_v    = nn.LayerNorm(dim)
        self.ln_t    = nn.LayerNorm(dim)
        self.sdrop_v = StochasticDepth(drop_path)
        self.sdrop_t = StochasticDepth(drop_path)

    def forward(self, V, T, text_key_padding_mask=None):
        # Vision attends to Text
        v_out, _ = self.v2t(V, T, T, key_padding_mask=text_key_padding_mask)
        V = self.ln_v(V + self.sdrop_v(v_out))
        # Text attends to (updated) Vision
        t_out, _ = self.t2v(T, V, V)
        T = self.ln_t(T + self.sdrop_t(t_out))
        return V, T


class FusionModule(nn.Module):
    """
    1. Project CLIP vision tokens [B, 196, 768] → [B, 196, FUSION_DIM]
    2. Project ViLT text tokens   [B,  40, 768] → [B,  40, FUSION_DIM]
    3. Run FUSION_LAYERS bidirectional cross-attention layers
    4. Mean-pool each stream and concatenate
    Output: [B, FUSION_DIM * 2] = [B, 512]
    """
    def __init__(self,
                 vision_dim=CLIP_VISION_DIM,
                 text_dim=VILT_TEXT_DIM,
                 fusion_dim=FUSION_DIM,
                 heads=FUSION_HEADS,
                 n_layers=FUSION_LAYERS,
                 dropout=DROPOUT_RATE):
        super().__init__()
        self.proj_v  = nn.Linear(vision_dim, fusion_dim)
        self.proj_t  = nn.Linear(text_dim,   fusion_dim)
        self.layers  = nn.ModuleList([
            CrossAttentionLayer(fusion_dim, heads, dropout, drop_path=0.10)
            for _ in range(n_layers)
        ])
        self.out_dim = fusion_dim * 2   # 512

    def forward(self, vision_tokens, text_tokens, text_pad_mask=None):
        V = self.proj_v(vision_tokens)   # [B, 196, 256]
        T = self.proj_t(text_tokens)     # [B,  40, 256]
        for layer in self.layers:
            V, T = layer(V, T, text_pad_mask)
        v_pooled = V.mean(dim=1)         # [B, 256]
        t_pooled = T.mean(dim=1)         # [B, 256]
        return torch.cat([v_pooled, t_pooled], dim=1)   # [B, 512]


print("FusionModule defined ✓")

# ── Cell 6: CLIP × ViLT Classifier ────────────────────────────────────────────
#
# BACKBONE UNFREEZING STRATEGY:
#   • Freeze all backbone params first
#   • Unfreeze ONLY the last UNFREEZE_BLOCKS=2 transformer blocks of each
#   • These fine-tune at LR_UNFREEZE=5e-6 (much smaller than head LR)
#   • This breaks the accuracy ceiling of the original fully-frozen design
#     while keeping overfitting under control

def _unfreeze_last_n(layer_list, n):
    for layer in list(layer_list)[-n:]:
        for p in layer.parameters():
            p.requires_grad = True


class CLIPViLTClassifier(nn.Module):
    def __init__(self, clip_model, vilt_model,
                 num_classes=NUM_CLASSES,
                 dropout=DROPOUT_RATE):
        super().__init__()
        self.clip = clip_model
        self.vilt = vilt_model

        # Step 1 — freeze all backbone parameters
        for p in self.clip.parameters(): p.requires_grad = False
        for p in self.vilt.parameters(): p.requires_grad = False

        # Step 2 — selectively unfreeze last N transformer blocks
        _unfreeze_last_n(self.clip.vision_model.encoder.layers, UNFREEZE_BLOCKS)
        for p in self.clip.vision_model.post_layernorm.parameters():
            p.requires_grad = True

        _unfreeze_last_n(self.vilt.encoder.layer, UNFREEZE_BLOCKS)
        for p in self.vilt.layernorm.parameters():
            p.requires_grad = True

        # Step 3 — trainable modules
        self.fusion = FusionModule(dropout=dropout)

        fused_dim = self.fusion.out_dim
        # 512 → 192 → 2
        self.classifier = nn.Sequential(
            nn.LayerNorm(fused_dim),
            nn.Dropout(dropout),
            nn.Linear(fused_dim, 192),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(192, num_classes),
        )

    def forward(self,
                clip_pixel_values,
                input_ids, token_type_ids, attention_mask,
                vilt_pixel_values,
                pixel_mask=None):

        # CLIP vision — patch tokens, drop CLS (position 0)
        clip_out      = self.clip.vision_model(
            pixel_values=clip_pixel_values,
            output_hidden_states=False, return_dict=True)
        vision_tokens = clip_out.last_hidden_state[:, 1:, :]  # [B, 196, 768]

        # ViLT — text token slice from joint embedding
        vilt_kwargs = dict(
            input_ids=input_ids, token_type_ids=token_type_ids,
            attention_mask=attention_mask, pixel_values=vilt_pixel_values)
        if pixel_mask is not None:
            vilt_kwargs["pixel_mask"] = pixel_mask
        vilt_out    = self.vilt(**vilt_kwargs, return_dict=True)
        n_text      = input_ids.shape[1]
        text_tokens = vilt_out.last_hidden_state[:, :n_text, :]  # [B, 40, 768]

        # Padding mask: True = padding token (should be ignored in attention)
        text_pad_mask = (attention_mask == 0)

        # Cross-attention fusion
        fused = self.fusion(vision_tokens, text_tokens, text_pad_mask)  # [B, 512]

        return self.classifier(fused)   # [B, 2]


def count_trainable(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen    = total - trainable
    print(f"  Params — total: {total:,} | trainable: {trainable:,} "
          f"({100*trainable/total:.2f}%) | frozen: {frozen:,}")


print("CLIPViLTClassifier (2-way, no metadata) defined ✓")

# ── Cell 7: Focal Loss + Balanced Weights ─────────────────────────────────────
#
# WHY FOCAL LOSS:
#   CrossEntropyLoss treats all samples equally — easy correct predictions
#   dominate the gradient signal. FocalLoss(γ=2) down-weights easy examples
#   so the model focuses on the hard boundary cases.
#
# WHY BALANCED CLASS WEIGHTS:
#   Even with 50/50 class split, focal loss alone may still produce a small
#   bias. Adding balanced class weights ensures Precision ≈ Recall ≈ F1 ≈ Acc.

class FocalLoss(nn.Module):
    def __init__(self, gamma=1.0, smoothing=LABEL_SMOOTH, weight=None):
        super().__init__()
        self.gamma = gamma
        self.ce    = nn.CrossEntropyLoss(
            weight=weight, label_smoothing=smoothing, reduction="none")

    def forward(self, logits, targets):
        ce = self.ce(logits, targets)
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()


def make_criterion(labels_arr, device):
    cw  = compute_class_weight(
        "balanced", classes=np.array([0, 1]), y=labels_arr)
    cwt = torch.tensor(cw, dtype=torch.float32).to(device)
    return FocalLoss(gamma=1.0, smoothing=LABEL_SMOOTH, weight=cwt)


print("Regularized loss (balanced 2-way) defined ✓")

# ── Cell 8: Training Helpers ───────────────────────────────────────────────────

class EarlyStopping:
    """Track val macro-F1 (higher = better). Save best checkpoint."""
    def __init__(self, patience=PATIENCE, delta=1e-4):
        self.patience   = patience
        self.delta      = delta
        self.best_score = None
        self.counter    = 0
        self.best_state = None
        self.stop       = False

    def step(self, score, model):
        if self.best_score is None or score > self.best_score + self.delta:
            self.best_score = score
            self.best_state = {k: v.cpu().clone()
                               for k, v in model.state_dict().items()}
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True

    def restore_best(self, model):
        if self.best_state:
            model.load_state_dict(
                {k: v.to(DEVICE) for k, v in self.best_state.items()})


def _to_device(batch):
    labels = batch.pop("labels").to(DEVICE)
    return {k: v.to(DEVICE) for k, v in batch.items()}, labels


def _forward(model, batch):
    b, labels = _to_device(batch)
    logits = model(
        clip_pixel_values = b["clip_pixel_values"],
        input_ids         = b["input_ids"],
        token_type_ids    = b["token_type_ids"],
        attention_mask    = b["attention_mask"],
        vilt_pixel_values = b["vilt_pixel_values"],
        pixel_mask        = b.get("pixel_mask"),
    )
    return logits, labels


def train_epoch(model, loader, optimizer, criterion, scaler, accum, scheduler=None):
    model.train()
    total_loss = correct = n = 0
    nb = len(loader)
    optimizer.zero_grad()

    for step, batch in enumerate(loader, 1):
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits, labels = _forward(model, batch)
            loss = criterion(logits, labels) / accum

        scaler.scale(loss).backward()

        if step % accum == 0 or step == nb:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            if scheduler is not None:
                scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * accum * labels.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        n          += labels.size(0)

        if step % 30 == 0 or step == nb:
            print(f"    step {step:3d}/{nb} | "
                  f"loss={total_loss/n:.4f} | acc={correct/n:.4f}", flush=True)

    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    preds, trues = [], []

    for batch in loader:
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits, labels = _forward(model, batch)
            loss = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        preds.extend(logits.argmax(1).cpu().numpy())
        trues.extend(labels.cpu().numpy())

    n = len(trues)
    return dict(
        loss      = total_loss / n,
        accuracy  = accuracy_score(trues, preds),
        f1        = f1_score(trues, preds, average="macro", zero_division=0),
        precision = precision_score(trues, preds, average="macro", zero_division=0),
        recall    = recall_score(trues, preds, average="macro", zero_division=0),
    ), np.array(preds), np.array(trues)


print("Training helpers ready ✓")

# ── Cell 9: 10-Fold Cross-Validation — 80 / 10 / 10 ──────────────────────────
#
# HOW THE 80/10/10 SPLIT WORKS WITH GROUPED CV:
#   StratifiedGroupKFold produces train+val / test folds while keeping duplicate
#   normalized headlines in only one side. The train+val side is then split into
#   train / validation by the same title groups.

def run_kfold(df):
    skf     = StratifiedGroupKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=SEED)
    labels  = df["label"].values
    groups  = df["split_group"].values
    indices = np.arange(len(df))

    fold_results, all_history, all_cms = [], [], []

    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        free, total = torch.cuda.mem_get_info()
        print(f"GPU before backbone load: {free/1e9:.1f}/{total/1e9:.1f} GB free")

    # Load both backbones once on CPU; copy state per fold to avoid contamination
    print(f"\nLoading CLIP ViT-B/32 ({CLIP_MODEL}) ...")
    clip_proc  = CLIPProcessor.from_pretrained(CLIP_MODEL)
    clip_base  = CLIPModel.from_pretrained(CLIP_MODEL).cpu()
    clip_state = {k: v.cpu().clone() for k, v in clip_base.state_dict().items()}
    n_clip     = sum(p.numel() for p in clip_base.parameters())
    print(f"  ✓ CLIP: {n_clip/1e6:.0f} M params")

    print(f"\nLoading ViLT-B/32 ({VILT_MODEL}) ...")
    vilt_proc  = ViltProcessor.from_pretrained(VILT_MODEL)
    vilt_base  = ViltModel.from_pretrained(VILT_MODEL).cpu()
    vilt_state = {k: v.cpu().clone() for k, v in vilt_base.state_dict().items()}
    n_vilt     = sum(p.numel() for p in vilt_base.parameters())
    print(f"  ✓ ViLT: {n_vilt/1e6:.0f} M params")

    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    for fold, (tv_idx, test_idx) in enumerate(skf.split(indices, labels, groups), 1):
        print(f"\n{'='*70}")
        print(f"  FOLD {fold} / {NUM_FOLDS}")
        print(f"{'='*70}")

        # ── 80 / 10 / 10 split ────────────────────────────────────────────────
        # Split validation by group too, so duplicate headlines never cross sets.
        tv_groups = groups[tv_idx]
        unique_groups = pd.Series(tv_groups).drop_duplicates().values
        group_labels = (
            df.iloc[tv_idx]
              .groupby("split_group")["label"]
              .agg(lambda s: int(s.mode().iloc[0]))
              .reindex(unique_groups)
              .values
        )
        train_groups, val_groups = train_test_split(
            unique_groups,
            test_size=1 / 9,
            random_state=SEED + fold,
            stratify=group_labels,
        )
        train_group_set = set(train_groups)
        val_group_set = set(val_groups)
        train_idx = tv_idx[np.isin(tv_groups, list(train_group_set))]
        val_idx = tv_idx[np.isin(tv_groups, list(val_group_set))]

        tr_pct  = 100 * len(train_idx) / len(df)
        vl_pct  = 100 * len(val_idx)   / len(df)
        te_pct  = 100 * len(test_idx)  / len(df)
        print(f"  Train:{len(train_idx):,} ({tr_pct:.0f}%) | "
              f"Val:{len(val_idx):,} ({vl_pct:.0f}%) | "
              f"Test:{len(test_idx):,} ({te_pct:.0f}%)")

        assert len(set(train_idx) & set(val_idx))  == 0, "Train/Val overlap!"
        assert len(set(train_idx) & set(test_idx)) == 0, "Train/Test overlap!"
        assert len(set(val_idx)   & set(test_idx)) == 0, "Val/Test overlap!"
        assert set(groups[train_idx]).isdisjoint(groups[val_idx]), "Train/Val group overlap!"
        assert set(groups[train_idx]).isdisjoint(groups[test_idx]), "Train/Test group overlap!"
        assert set(groups[val_idx]).isdisjoint(groups[test_idx]), "Val/Test group overlap!"

        # Per-fold class-balanced criterion
        criterion = make_criterion(labels[train_idx], DEVICE)

        # DataLoaders
        nw  = 2 if DEVICE == "cuda" else 0
        pin = DEVICE == "cuda"

        def make_loader(idx, shuffle):
            ds = FakedditDataset(df.iloc[idx], clip_proc, vilt_proc)
            return DataLoader(ds, BATCH_SIZE, shuffle=shuffle,
                              collate_fn=collate_fn,
                              num_workers=nw, pin_memory=pin)

        train_loader = make_loader(train_idx, True)
        val_loader   = make_loader(val_idx,   False)
        test_loader  = make_loader(test_idx,  False)

        # Reset backbone weights to fresh state (no inter-fold contamination)
        print("  Initialising model ...")
        clip_base.load_state_dict(clip_state, strict=True)
        vilt_base.load_state_dict(vilt_state, strict=True)
        clip_base.to(DEVICE)
        vilt_base.to(DEVICE)

        if DEVICE == "cuda":
            free, total = torch.cuda.mem_get_info()
            print(f"  GPU after backbone → GPU: {free/1e9:.1f}/{total/1e9:.1f} GB free")

        model = CLIPViLTClassifier(clip_base, vilt_base).to(DEVICE)
        count_trainable(model)

        # Layer-wise LR: backbone unfrozen blocks get tiny LR, head gets full LR
        bb_params   = [p for n, p in model.named_parameters()
                       if p.requires_grad and ("clip." in n or "vilt." in n)]
        head_params = [p for n, p in model.named_parameters()
                       if p.requires_grad and "clip." not in n and "vilt." not in n]

        optimizer = AdamW([
            {"params": head_params, "lr": LR_HEAD,     "weight_decay": WEIGHT_DECAY},
            {"params": bb_params,   "lr": LR_UNFREEZE, "weight_decay": WEIGHT_DECAY},
        ], eps=1e-8)

        # Cosine schedule with linear warm-up
        total_steps  = max(1, int(np.ceil(len(train_loader) / ACCUM_STEPS)) * EPOCHS)
        warmup_steps = max(1, int(WARMUP_RATIO * total_steps))
        scheduler    = get_cosine_schedule_with_warmup(
            optimizer, warmup_steps, total_steps)

        early_stop = EarlyStopping(PATIENCE)
        history    = {
            "train_loss": [], "val_loss": [],
            "train_acc":  [], "val_acc":  [], "val_f1": [],
        }

        for epoch in range(1, EPOCHS + 1):
            print(f"\n  ── Epoch {epoch}/{EPOCHS} ────────────────────────────")
            tr_loss, tr_acc = train_epoch(
                model, train_loader, optimizer, criterion, scaler, ACCUM_STEPS, scheduler)

            val_m, _, _ = evaluate(model, val_loader, criterion)
            gap = max(0.0, tr_acc - val_m["accuracy"])
            monitor_score = val_m["f1"] - OVERFIT_PENALTY * gap
            marker = "  ← best" if early_stop.counter == 0 else ""

            print(f"  E{epoch:2d}/{EPOCHS} | "
                  f"TrAcc={tr_acc:.4f} | "
                  f"VaAcc={val_m['accuracy']:.4f}  "
                  f"VaF1={val_m['f1']:.4f}  "
                  f"VaPrec={val_m['precision']:.4f}  "
                  f"VaRec={val_m['recall']:.4f} | "
                  f"gap={gap:.4f}  score={monitor_score:.4f}{marker}")

            history["train_loss"].append(tr_loss)
            history["val_loss"].append(val_m["loss"])
            history["train_acc"].append(tr_acc)
            history["val_acc"].append(val_m["accuracy"])
            history["val_f1"].append(val_m["f1"])

            early_stop.step(monitor_score, model)
            if early_stop.stop:
                print(f"  ⏹  Early stop at epoch {epoch} "
                      f"(best val F1={early_stop.best_score:.4f})")
                break

        # ── Test evaluation (best checkpoint) ─────────────────────────────────
        early_stop.restore_best(model)
        test_m, preds, trues = evaluate(model, test_loader, criterion)
        cm      = confusion_matrix(trues, preds)
        ovf_gap = abs(max(history["train_acc"]) - test_m["accuracy"])
        all_cms.append(cm)

        print(f"\n  ── Fold {fold} Test Results ──────────────────────────────")
        print(f"  Accuracy  : {test_m['accuracy']:.4f}")
        print(f"  Macro-F1  : {test_m['f1']:.4f}")
        print(f"  Precision : {test_m['precision']:.4f}")
        print(f"  Recall    : {test_m['recall']:.4f}")
        print(f"  OvfGap    : {ovf_gap:.4f}"
              + ("  ✅" if ovf_gap < 0.05 else "  ⚠"))
        print()
        print(classification_report(
            trues, preds, target_names=["Real", "Fake"], digits=4))

        fold_results.append({
            "Fold"      : fold,
            "Accuracy"  : round(test_m["accuracy"],  4),
            "F1-Score"  : round(test_m["f1"],        4),
            "Precision" : round(test_m["precision"],  4),
            "Recall"    : round(test_m["recall"],     4),
            "OvfGap"    : round(ovf_gap,              4),
        })
        all_history.append(history)

        torch.save(model.state_dict(),
                   f"{OUTPUT_DIR}/clip_vilt_fold{fold}.pt")

        # Free GPU memory before next fold
        clip_base.to("cpu")
        vilt_base.to("cpu")
        del model, optimizer, scheduler, early_stop
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            free, total = torch.cuda.mem_get_info()
            print(f"  GPU after cleanup: {free/1e9:.1f}/{total/1e9:.1f} GB free")

    return fold_results, all_history, all_cms


print("10-fold runner (80/10/10) ready ✓")

# ── Cell 10: Results & Visualisation ──────────────────────────────────────────

def print_results_table(fold_results):
    print("\n" + "═" * 80)
    print("  📊 CLIP × ViLT — 2-WAY  (Real vs Fake)  —  No Metadata")
    print("═" * 80)
    rows  = [[r["Fold"], r["Accuracy"], r["F1-Score"],
               r["Precision"], r["Recall"], r["OvfGap"]]
             for r in fold_results]
    accs  = [r["Accuracy"]  for r in fold_results]
    f1s   = [r["F1-Score"]  for r in fold_results]
    precs = [r["Precision"] for r in fold_results]
    recs  = [r["Recall"]    for r in fold_results]
    gaps  = [r["OvfGap"]    for r in fold_results]
    rows.append([
        "Mean ± Std",
        f"{np.mean(accs):.4f} ± {np.std(accs):.4f}",
        f"{np.mean(f1s):.4f} ± {np.std(f1s):.4f}",
        f"{np.mean(precs):.4f} ± {np.std(precs):.4f}",
        f"{np.mean(recs):.4f} ± {np.std(recs):.4f}",
        f"{np.mean(gaps):.4f} ± {np.std(gaps):.4f}",
    ])
    print(tabulate(
        rows,
        headers=["Fold", "Accuracy", "F1-Score",
                 "Precision", "Recall", "OvfGap"],
        tablefmt="fancy_grid",
    ))
    print("═" * 80)
    mean_gap = np.mean(gaps)
    print(f"\n  ✅ Mean Accuracy  : {np.mean(accs):.4f}")
    print(f"  ✅ Mean Macro-F1  : {np.mean(f1s):.4f}")
    print(f"  ✅ Mean Precision : {np.mean(precs):.4f}")
    print(f"  ✅ Mean Recall    : {np.mean(recs):.4f}")
    print(f"  {'✅' if mean_gap < 0.05 else '⚠️ '} Mean OvfGap  : {mean_gap:.4f}"
          + ("  (< 5% ✅)" if mean_gap < 0.05 else "  (> 5% ⚠)"))
    return dict(mean_accuracy=np.mean(accs), mean_f1=np.mean(f1s),
                mean_precision=np.mean(precs), mean_recall=np.mean(recs))


def save_results_csv(fold_results):
    dfr  = pd.DataFrame(fold_results)
    path = f"{OUTPUT_DIR}/clip_vilt_2way_results.csv"
    dfr.to_csv(path, index=False)
    print(f"✓ Results saved: {path}")
    print(dfr.to_string(index=False))


def plot_visualizations(all_history, all_cms):
    nf = len(all_history)
    C  = ["#2196F3", "#FF5722"]

    # Training curves (2 rows × nf cols)
    fig, axes = plt.subplots(2, nf, figsize=(4 * nf, 8))
    if nf == 1:
        axes = axes.reshape(2, 1)

    for i, h in enumerate(all_history):
        ep = range(1, len(h["train_loss"]) + 1)

        # Loss
        ax = axes[0, i]
        ax.plot(ep, h["train_loss"], color=C[0], lw=2, label="Train")
        ax.plot(ep, h["val_loss"],   color=C[1], lw=2, ls="--", label="Val")
        ax.set_title(f"Fold {i+1} — Loss", fontweight="bold", fontsize=9)
        ax.legend(fontsize=7); ax.grid(alpha=0.3)

        # Accuracy + gap
        ax = axes[1, i]
        ax.plot(ep, h["train_acc"], color=C[0], lw=2, label="Train")
        ax.plot(ep, h["val_acc"],   color=C[1], lw=2, ls="--", label="Val")
        ax.axhline(0.90, color="green", ls=":", lw=1.5, label="Target 90%")
        ax.set_title(f"Fold {i+1} — Accuracy", fontweight="bold", fontsize=9)
        ax.set_ylim(0, 1); ax.legend(fontsize=7); ax.grid(alpha=0.3)
        gap = [abs(a - b) for a, b in zip(h["train_acc"], h["val_acc"])]
        ax2 = ax.twinx()
        ax2.plot(ep, gap, color="grey", lw=1, ls=":", alpha=0.7)
        ax2.set_ylabel("gap", fontsize=7, color="grey")
        ax2.set_ylim(0, 0.30)
        ax2.tick_params(labelsize=7)

    plt.suptitle(
        "CLIP × ViLT — 2-Way Fake News Detection  (No Metadata, Grouped CV)",
        fontsize=12, fontweight="bold", y=1.01)
    plt.tight_layout()
    path = f"{OUTPUT_DIR}/training_curves_2way.png"
    plt.savefig(path, dpi=130, bbox_inches="tight")
    plt.show()
    print(f"✓ Saved: {path}")

    # Aggregated confusion matrix
    agg_cm = np.sum(all_cms, axis=0)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(agg_cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Real", "Fake"],
                yticklabels=["Real", "Fake"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"Aggregated Confusion Matrix — All {NUM_FOLDS} Folds",
                 fontweight="bold")
    plt.tight_layout()
    path = f"{OUTPUT_DIR}/confusion_matrix_2way.png"
    plt.savefig(path, dpi=130, bbox_inches="tight")
    plt.show()
    print(f"✓ Saved: {path}")

    # Per-fold accuracy bar chart
    fold_accs = [h["val_acc"][-1] if h["val_acc"] else 0 for h in all_history]
    fig, ax   = plt.subplots(figsize=(max(8, nf * 0.9), 4))
    bars = ax.bar(range(1, nf + 1), fold_accs, color="steelblue", alpha=0.8)
    ax.axhline(0.90, color="green", ls=":", lw=1.5, label="Target 90%")
    ax.set_xlabel("Fold"); ax.set_ylabel("Val Accuracy")
    ax.set_ylim(0.5, 1.05)
    ax.set_title("Val Accuracy per Fold — CLIP × ViLT 2-Way No Metadata")
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.legend(); ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    path = f"{OUTPUT_DIR}/fold_accuracy_2way.png"
    plt.savefig(path, dpi=130, bbox_inches="tight")
    plt.show()
    print(f"✓ Saved: {path}")


print("Visualisation helpers ready ✓")

# ── Cell 11: Load Fakeddit ─────────────────────────────────────────────────────
print("Loading Fakeddit 2-way dataset (no metadata, grouped by normalized title)...\n")
df_main = load_fakeddit(NUM_SAMPLES)
print("\nSample rows:")
print(df_main[["text", "label", "image_path", "split_group"]].head(4).to_string(index=False))

# ── Cell 12: Run 10-Fold CV ────────────────────────────────────────────────────
# ⏱  Estimated runtime on Kaggle T4:  ~1.5 – 3 hours
#    (last 2 backbone blocks unfrozen — heavier than fully-frozen)
print(f"\nStarting {NUM_FOLDS}-fold CV  "
      f"[CLIP ViT-B/32 × ViLT-B/32 · 2-way · no metadata · {EPOCHS} epochs]...\n")
fold_results, all_history, all_cms = run_kfold(df_main)

# ── Cell 13: Results ───────────────────────────────────────────────────────────
summary = print_results_table(fold_results)
save_results_csv(fold_results)

# ── Cell 14: Visualisations ────────────────────────────────────────────────────
plot_visualizations(all_history, all_cms)

# ── Cell 15: Final Summary ─────────────────────────────────────────────────────
print("\n" + "═" * 70)
print("  🎯  FINAL SUMMARY  —  CLIP × ViLT  2-Way  v9")
print("═" * 70)
print(f"  Vision backbone : CLIP ViT-B/32  — last {UNFREEZE_BLOCKS} blocks unfrozen  (LR={LR_UNFREEZE})")
print(f"  Text   backbone : ViLT-B/32      — last {UNFREEZE_BLOCKS} blocks unfrozen  (LR={LR_UNFREEZE})")
print(f"  Fusion          : Bidirectional Cross-Attention")
print(f"                    dim={FUSION_DIM}  heads={FUSION_HEADS}  layers={FUSION_LAYERS}  + StochasticDepth")
print(f"  Metadata        : DISABLED (no subreddit, score, upvote_ratio, domain, comments, or text length)")
print(f"  Split           : Grouped by normalized title to block duplicate-headline leakage")
print(f"  Task            : 2-WAY  (Real vs Fake)  — 3-way & 6-way EXCLUDED")
print(f"  Dataset         : Fakeddit  {NUM_SAMPLES:,} samples  {NUM_FOLDS}-fold CV  80/10/10")
print(f"  Loss            : FocalLoss(γ=1) + balanced class weights + label smooth={LABEL_SMOOTH}")
print(f"  Regularisation  : dropout={DROPOUT_RATE} | wd={WEIGHT_DECAY} | grad_clip=1.0 | EarlyStop(F1)")
print(f"  Epochs          : {EPOCHS} max  (patience={PATIENCE})")
print()
print(f"  📊 Cross-Validated Performance:")
print(f"     Accuracy  : {summary['mean_accuracy']:.4f}")
print(f"     Macro-F1  : {summary['mean_f1']:.4f}")
print(f"     Precision : {summary['mean_precision']:.4f}")
print(f"     Recall    : {summary['mean_recall']:.4f}")
print()
print(f"  💾 Files saved to {OUTPUT_DIR}/")
print(f"     clip_vilt_2way_results.csv")
print(f"     training_curves_2way.png")
print(f"     confusion_matrix_2way.png")
print(f"     fold_accuracy_2way.png")
print(f"     clip_vilt_fold{{1..{NUM_FOLDS}}}.pt")
print("═" * 70)
print("  ✅  Training complete!")
print("═" * 70)

[1/5] transformers>=4.40.0 ... ✓
[2/5] pillow ... ✓
[3/5] tabulate ... ✓
[4/5] scikit-learn ... ✓
[5/5] timm ... ✓

transformers 5.0.0 | torch 2.10.0+cu128
CUDA: True
GPU : Tesla T4
VRAM: 15.6 GB
Device        : cuda
Task          : 2-WAY ONLY  (3-way & 6-way EXCLUDED)
Samples       : 40,000  (balanced)
CV            : 5-fold grouped by normalized title  |  Split: 80 / 10 / 10
Epochs        : 10  (EarlyStopping patience=3)
Metadata      : DISABLED  (headline text + image pixels only)
Backbone      : CLIP ViT-B/32 + ViLT-B/32
Unfreeze      : last 1 blocks per backbone  LR=2e-06
Fusion        : 256-d  4h  1L + StochasticDepth
Loss          : CrossEntropy + class weights + label smoothing
Fakeddit loader (2-way, no metadata) ready ✓
FakedditDataset (2-way text + image only) defined ✓
FusionModule defined ✓
CLIPViLTClassifier (2-way, no metadata) defined ✓
Regularized loss (balanced 2-way) defined ✓
Training helpers ready ✓
10-fold runner (80/10/10) ready ✓
Visualisation helpers ready ✓
Lo

GPU before backbone load: 15.5/15.6 GB free

Loading CLIP ViT-B/32 (openai/clip-vit-base-patch32) ...


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

ViltModel LOAD REPORT from: dandelin/vilt-b32-mlm
Key                                          | Status     |  | 
---------------------------------------------+------------+--+-
mlm_score.transform.dense.weight             | UNEXPECTED |  | 
vilt.embeddings.text_embeddings.position_ids | UNEXPECTED |  | 
mlm_score.decoder.weight                     | UNEXPECTED |  | 
mlm_score.transform.LayerNorm.bias           | UNEXPECTED |  | 
mlm_score.transform.dense.bias               | UNEXPECTED |  | 
mlm_score.transform.LayerNorm.weight         | UNEXPECTED |  | 
mlm_score.bias                               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

  ✓ ViLT: 112 M params

  FOLD 1 / 5
  Train:27,973 (70%) | Val:3,521 (9%) | Test:8,506 (21%)
  Initialising model ...
  GPU after backbone → GPU: 14.4/15.6 GB free
  Params — total: 263,893,315 | trainable: 15,199,810 (5.76%) | frozen: 248,693,505

  ── Epoch 1/10 ────────────────────────────
    step  30/1749 | loss=0.3787 | acc=0.5208
    step  60/1749 | loss=0.3821 | acc=0.5052
    step  90/1749 | loss=0.3869 | acc=0.5000
    step 120/1749 | loss=0.3868 | acc=0.4995
    step 150/1749 | loss=0.3845 | acc=0.5075
    step 180/1749 | loss=0.3828 | acc=0.5097
    step 210/1749 | loss=0.3822 | acc=0.5122
    step 240/1749 | loss=0.3797 | acc=0.5146
    step 270/1749 | loss=0.3765 | acc=0.5213
    step 300/1749 | loss=0.3767 | acc=0.5212
    step 330/1749 | loss=0.3759 | acc=0.5252
    step 360/1749 | loss=0.3733 | acc=0.5285
    step 390/1749 | loss=0.3718 | acc=0.5298
    step 420/1749 | loss=0.3708 | acc=0.5330
    step 450/1749 | loss=0.3703 | acc=0.5329
    step 480/1749 | loss=0.369

KeyboardInterrupt: 